# 09 — Layers and the forward pass

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/009-layers-and-forward-pass.ipynb)

A small network is just a few neurons stacked, and the forward pass is the arithmetic that turns an input into an output. Here you compute a 2-2-1 network by hand, draw what each hidden unit does to the input plane, and then remove the bend to see the whole thing collapse.

## 1. The numbers

Two inputs feed two hidden units, and those two units feed one output. Each arrow carries a weight, each unit carries a bias, and the hidden units bend their scores with ReLU before passing them on.

In [ ]:
import numpy as np                                    # the array library, preinstalled on Colab

x = np.array([1.0, -1.0])                             # the input: two features
W1 = np.array([[1.0, 0.5], [-1.0, 2.0]])              # layer 1: row per hidden unit, column per input
b1 = np.array([0.0, -0.5])                            # layer 1: one bias per hidden unit
W2 = np.array([[1.0, -2.0]])                          # layer 2: one output unit, two weights
b2 = 0.25                                             # layer 2: a single bias

z1 = W1 @ x + b1                                      # hidden scores, before any bend
a1 = np.maximum(0.0, z1)                              # hidden activations, after ReLU
z2 = (W2 @ a1)[0] + b2                                # the network's single output
print("z1 =", z1)                                     # [0.5, -3.5]
print("a1 =", a1)                                     # [0.5, 0.0]: the second unit is clipped
print("z2 =", z2)                                     # 0.75

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads NumPy, the array library Colab already has.</li><li><code>x = ...</code> — the input vector, two features.</li><li><code>W1 = ...</code> — the first weight matrix; each row belongs to one hidden unit.</li><li><code>b1 = ...</code> — one bias per hidden unit.</li><li><code>W2 = ...</code> — the second weight matrix; it produces the single output.</li><li><code>b2 = 0.25</code> — the output bias.</li><li><code>z1 = W1 @ x + b1</code> — matrix times vector, then the bias: two hidden scores.</li><li><code>a1 = ...</code> — ReLU applied elementwise to both hidden scores.</li><li><code>z2 = (W2 @ a1)[0] + b2</code> — dot the two activations with the output weights.</li><li><code>print("z1 = ...")</code> — the hidden scores before the bend.</li><li><code>print("a1 = ...")</code> — the hidden activations after the bend.</li><li><code>print("z2 = ...")</code> — the network's single output.</li></ul>

</details>

## 2. Step by step

The matrix product hides two separate dot products. Written out, each hidden unit takes both inputs, weights them, adds its bias, and then the output unit does the same to the two activations.

In [ ]:
def forward(x, W1, b1, W2, b2):                       # the whole 2-2-1 network in one call
    z1 = W1 @ x + b1                                  # hidden scores: two numbers
    a1 = np.maximum(0.0, z1)                          # hidden activations: the bend
    z2 = (W2 @ a1)[0] + b2                            # output: one number
    return z1, a1, z2                                 # hand back every stage

print("step 1 - hidden raw scores z1:")               # label the stage
for k in range(2):                                    # one line per hidden unit
    print(f"  unit {k+1}: {W1[k,0]:+.2f}*{x[0]:+.2f} + {W1[k,1]:+.2f}*{x[1]:+.2f} + {b1[k]:+.2f} = {z1[k]:+.2f}")   # expand the dot product
print("step 2 - after ReLU, a1 =", np.maximum(0.0, z1))   # negatives become zero
print("step 3 - output z2 =", (W2 @ np.maximum(0.0, z1))[0] + b2)   # second dot product
print("step 4 - throughput of the output unit:")      # label the last stage
for k in range(2):                                    # one term per hidden unit
    print(f"  {W2[0,k]:+.2f} * {a1[k]:+.2f}")        # each unit's contribution to z2
print("check - forward() agrees:", forward(x, W1, b1, W2, b2)[2])   # 0.75, same as above

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>def forward(x, W1, b1, W2, b2)</code> — the full pass, reusable.</li><li><code>z1 = W1 @ x + b1</code> — the hidden layer's scores.</li><li><code>a1 = np.maximum(0.0, z1)</code> — the ReLU bend.</li><li><code>z2 = (W2 @ a1)[0] + b2</code> — the output layer; <code>[0]</code> unwraps the length-1 array.</li><li><code>return z1, a1, z2</code> — every stage, so any of them can be inspected.</li><li><code>print("step 1 ...")</code> — labels the first stage.</li><li><code>for k in range(2)</code> — walk the two hidden units.</li><li><code>print(f"  unit {k+1}: ...")</code> — spell out the two products and the bias.</li><li><code>print("step 2 ...")</code> — the two activations after clipping.</li><li><code>print("step 3 ...")</code> — the same output recomputed in one expression.</li><li><code>print("step 4 ...")</code> — labels the output unit's terms.</li><li><code>for k in range(2)</code> — walk the two contributions again.</li><li><code>print(f"  {W2[0,k]:+.2f} * ...")</code> — each hidden unit's share of the output.</li><li><code>forward(x, ...)[2]</code> — index 2 picks the output from the returned triple.</li></ul>

</details>

## 3. Plot: what each hidden unit sees

Run the hidden layer over a grid of inputs and draw each unit's activation as a colour map. One unit is on for some inputs and off for others, and the boundary between those two regions is a straight line set by its weights.

In [ ]:
import matplotlib.pyplot as plt                        # plotting library, preinstalled on Colab

grid = np.linspace(-2, 2, 41)                         # 41 input values per axis
heat = np.zeros((2, len(grid), len(grid)))            # two units, one plane each
for i, x1 in enumerate(grid):                         # walk the first input
    for j, x2 in enumerate(grid):                     # walk the second input
        zz = W1 @ np.array([x1, x2]) + b1             # forward pass into the hidden layer
        heat[:, i, j] = np.maximum(0.0, zz)           # store both ReLU'd values

fig, axes = plt.subplots(1, 2, figsize=(8, 3))        # two panels, one per hidden unit
for k in range(2):                                    # draw each unit's map
    im = axes[k].imshow(heat[k], origin="lower", extent=[-2, 2, -2, 2], cmap="viridis")   # the activation plane
    axes[k].set_title(f"hidden unit {k+1}")           # which unit this is
    axes[k].set_xlabel("x1")                          # x-axis label
    axes[k].set_ylabel("x2")                          # y-axis label
    fig.colorbar(im, ax=axes[k])                      # the colour scale for this panel
plt.suptitle("each hidden unit splits the plane along its own line")   # the takeaway
plt.show()                                            # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — Colab already ships the plotting library.</li><li><code>grid = np.linspace(-2, 2, 41)</code> — the input values used on both axes.</li><li><code>heat = np.zeros((2, 41, 41))</code> — an empty box for two activation maps.</li><li><code>for i, x1 in enumerate(grid)</code> — the outer loop over the first input.</li><li><code>for j, x2 in enumerate(grid)</code> — the inner loop over the second input.</li><li><code>zz = W1 @ np.array([x1, x2]) + b1</code> — one forward pass for this grid point.</li><li><code>heat[:, i, j] = ...</code> — write both hidden values into their maps.</li><li><code>plt.subplots(1, 2, ...)</code> — two panels side by side.</li><li><code>for k in range(2)</code> — one pass per hidden unit.</li><li><code>axes[k].imshow(...)</code> — the map, with <code>origin="lower"</code> so up is up.</li><li><code>axes[k].set_title(...)</code> — says which unit the panel belongs to.</li><li><code>axes[k].set_xlabel("x1")</code> — names the first input axis.</li><li><code>axes[k].set_ylabel("x2")</code> — names the second input axis.</li><li><code>fig.colorbar(...)</code> — the colour scale, so the brightness has units.</li><li><code>plt.suptitle(...)</code> — the takeaway, above both panels.</li><li><code>plt.show()</code> — draws the figure.</li></ul>

</details>

## 4. Interactive: move the hidden bias

The bias sets where the bend starts. Drag it and the two hidden scores slide up or down together; the readout reports the output, which changes only while a unit is above zero.

In [ ]:
from ipywidgets import interact                        # sliders that rerun a function

def show(b1_value=0.0):                               # b1_value comes from the slider
    b1_i = np.array([b1_value, b1_value])             # both units get the same shift
    z1_i = W1 @ x + b1_i                              # hidden scores at this bias
    a1_i = np.maximum(0.0, z1_i)                      # hidden activations
    z2_i = (W2 @ a1_i)[0] + b2                        # the output
    plt.figure(figsize=(6, 3))                        # fresh figure on every drag
    plt.bar([0, 1], z1_i, width=0.35, color="tab:grey", label="z1 before ReLU")   # raw scores
    plt.bar([0.35, 1.35], a1_i, width=0.35, color="tab:blue", label="a1 after ReLU")   # activations
    plt.axhline(0, color="grey", linewidth=0.6)       # the zero line
    plt.xticks([0.175, 1.175], ["unit 1", "unit 2"])  # label each pair of bars
    plt.ylim(-5, 3)                                   # fixed axis, so nothing jumps
    plt.xlabel("hidden unit")                         # x-axis label
    plt.ylabel("value")                               # y-axis label
    plt.title(f"b1 = {b1_value:+.2f} -> z2 = {z2_i:+.2f}")   # live readout
    plt.legend()                                      # name the bars
    plt.show()                                        # draw it

interact(show, b1_value=(-2.0, 2.0, 0.5))             # attach the slider

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — Colab ships the widgets library.</li><li><code>def show(b1_value=0.0)</code> — the function that redraws one frame.</li><li><code>b1_i = ...</code> — the slider value, copied into both hidden biases.</li><li><code>z1_i = W1 @ x + b1_i</code> — the hidden scores at this bias.</li><li><code>a1_i = ...</code> — the same scores after ReLU.</li><li><code>z2_i = ...</code> — the output, recomputed so the title can report it.</li><li><code>plt.figure(...)</code> — a new canvas for each frame.</li><li><code>plt.bar([0, 1], z1_i, ...)</code> — the raw scores, in grey.</li><li><code>plt.bar([0.35, 1.35], a1_i, ...)</code> — the activations, in blue.</li><li><code>plt.axhline(0, ...)</code> — the zero line.</li><li><code>plt.xticks(...)</code> — names the units under their bars.</li><li><code>plt.ylim(-5, 3)</code> — a fixed range, so the bars do not jump.</li><li><code>plt.xlabel(...)</code> — names the hidden units.</li><li><code>plt.ylabel(...)</code> — names the measured value.</li><li><code>plt.title(f"...")</code> — the bias and the resulting output, live.</li><li><code>plt.legend()</code> — matches each bar to its label.</li><li><code>plt.show()</code> — draws the frame.</li><li><code>interact(show, b1_value=(...))</code> — the slider, its range and its step.</li></ul>

</details>

## 5. When it breaks: no bend means one layer

Remove the ReLU and the two layers multiply out into a single weight matrix, so the network can only draw straight lines. Below, the bent network and its flattened copy are plotted on the same axes while one input sweeps from -3 to 3.

In [ ]:
W_combined = W2 @ W1                                  # two linear layers fused into one
print("W2 @ W1 =", W_combined)                        # a single 1x2 row: the hidden width vanished

path = np.linspace(-3, 3, 121)                        # sweep one input, hold the other at 0
inp = np.stack([path, np.zeros_like(path)])           # shape (2, 121): one column per test point
z_lin = (W2 @ np.maximum(0.0, W1 @ inp + b1[:, None]))[0] + b2   # the real net, with the bend
z_flat = (W_combined @ inp)[0] + (W2 @ b1)[0] + b2    # the collapsed net: one straight line

plt.figure(figsize=(6, 3))                            # new figure
plt.plot(path, z_lin, color="tab:blue", linewidth=2, label="2 layers, with ReLU")   # bends at a corner
plt.plot(path, z_flat, color="tab:red", linestyle="--", label="W2 @ W1, no ReLU")   # one straight line
plt.xlabel("x1 (x2 fixed at 0)")                      # x-axis label
plt.ylabel("network output z2")                       # y-axis label
plt.title("without the bend, two layers are just one layer")   # the failure
plt.legend()                                          # name the curves
plt.show()                                            # render figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>W_combined = W2 @ W1</code> — matrix multiply fuses the two linear maps.</li><li><code>print("W2 @ W1 = ...")</code> — the fused map has one row and two columns.</li><li><code>path = ...</code> — 121 values to sweep the first input.</li><li><code>inp = np.stack(...)</code> — a 2-by-121 batch, with the second input pinned at zero.</li><li><code>z_lin = ...</code> — the real network's output: hidden scores, the bend, then layer two.</li><li><code>z_flat = ...</code> — the fused network; <code>(W2 @ b1)[0] + b2</code> folds the biases in too.</li><li><code>plt.figure(...)</code> — a new canvas.</li><li><code>plt.plot(path, z_lin, ...)</code> — the bent curve, in blue.</li><li><code>plt.plot(path, z_flat, ...)</code> — the straight line, dashed red.</li><li><code>plt.xlabel(...)</code> — says which input was swept.</li><li><code>plt.ylabel(...)</code> — names the output.</li><li><code>plt.title(...)</code> — states why the collapse matters.</li><li><code>plt.legend()</code> — matches each curve to its label.</li><li><code>plt.show()</code> — draws the figure.</li></ul>

</details>

## Your turn

In section 4, drag `b1` until hidden unit 2 leaves the flat part of ReLU, and note the output before and after. Then change `W2` in section 1 to `[[1.0, 0.0]]` and predict how `z2` depends on the second hidden unit.